# HW#9 – Part 2: Text Processing (encoding and decoding)
**Course:** ECE-5831 &nbsp;|&nbsp; **Notebook:** `Module_9_2.ipynb`

## Goal
Neural networks cannot read raw text – they need **numbers**. This notebook implements the class `TextProcess`, which

1. **encodes** text into numerical representations (integer sequences, padded batches, one-hot vectors), and
2. **decodes** those numbers back into readable text,

and then **tests** that the round trip `text → numbers → text` works.

## Requirements checklist
| Requirement | Where |
|---|---|
| `TextProcess` – text **encoding** | `fit`, `encode`, `encode_batch`, `one_hot_encode` |
| `TextProcess` – text **decoding** | `decode`, `decode_batch`, `one_hot_decode` |
| Demonstrate text → numbers → readable text | Sections 3–6 |
| Tests that all methods work | Section 7 (all assertions print `PASS`) |

## 1. Imports

In [1]:
import re, json
from collections import Counter
import numpy as np
import pandas as pd

## 2. The `TextProcess` class
**How it works**

* **Tokenisation** – the text is split into tokens: *words* (`mode="word"`, punctuation marks become their own tokens and contractions such as `don't` stay together) or single *characters* (`mode="char"`).
* **Vocabulary** – `fit()` counts the tokens of a training corpus and gives the most frequent token the smallest integer. Two indices are reserved: `0` = `<PAD>` (padding) and `1` = `<OOV>` (words never seen during `fit`).
* **Encoding** – `encode()` maps every token to its integer index; `encode_batch()` pads many sequences to one length; `one_hot_encode()` turns indices into one-hot vectors.
* **Decoding** – `decode()` maps indices back to tokens and re-joins them into a readable sentence (spaces before punctuation are removed); `one_hot_decode()` uses `argmax` to recover the indices first.

In [2]:
class TextProcess:
    """Encode text into integer / one-hot representations and decode it back."""

    PAD, OOV = "<PAD>", "<OOV>"
    TOKEN_RE = re.compile(r"\w+(?:'\w+)?|[^\w\s]")      # words (keeping don't), or one punctuation mark

    def __init__(self, mode="word", lower=True, num_words=None):
        assert mode in ("word", "char"), "mode must be 'word' or 'char'"
        self.mode, self.lower, self.num_words = mode, lower, num_words
        self.word_index = {self.PAD: 0, self.OOV: 1}      # token -> id
        self.index_word = {0: self.PAD, 1: self.OOV}      # id -> token
        self.word_counts = Counter()

    # ---------- preprocessing ----------
    def normalize(self, text):
        """Lower-case (optional) and collapse repeated white space."""
        text = text.lower() if self.lower else text
        return re.sub(r"\s+", " ", text).strip() if self.mode == "word" else text

    def tokenize(self, text):
        text = self.normalize(text)
        return self.TOKEN_RE.findall(text) if self.mode == "word" else list(text)

    # ---------- vocabulary ----------
    def fit(self, texts):
        """Build the vocabulary from a list of texts (most frequent token -> smallest id)."""
        self.word_counts = Counter(tok for t in texts for tok in self.tokenize(t))
        most_common = self.word_counts.most_common(self.num_words)
        self.word_index = {self.PAD: 0, self.OOV: 1}
        for tok, _ in most_common:
            self.word_index.setdefault(tok, len(self.word_index))
        self.index_word = {i: t for t, i in self.word_index.items()}
        return self

    @property
    def vocab_size(self):
        return len(self.word_index)

    # ---------- ENCODING ----------
    def encode(self, text):
        """text -> list of integer ids (unknown tokens -> OOV id)."""
        oov = self.word_index[self.OOV]
        return [self.word_index.get(tok, oov) for tok in self.tokenize(text)]

    def encode_batch(self, texts, maxlen=None, padding="post"):
        """list of texts -> 2-D numpy array of ids, padded (with 0) / truncated to maxlen."""
        seqs = [self.encode(t) for t in texts]
        maxlen = maxlen or max(len(s) for s in seqs)
        out = np.zeros((len(seqs), maxlen), dtype="int32")
        for i, s in enumerate(seqs):
            s = s[:maxlen]
            if padding == "post":
                out[i, :len(s)] = s
            else:
                out[i, maxlen - len(s):] = s
        return out

    def one_hot_encode(self, text_or_ids):
        """text (or list of ids) -> (sequence_length, vocab_size) one-hot matrix."""
        ids = self.encode(text_or_ids) if isinstance(text_or_ids, str) else list(text_or_ids)
        m = np.zeros((len(ids), self.vocab_size), dtype="float32")
        m[np.arange(len(ids)), ids] = 1.0
        return m

    # ---------- DECODING ----------
    def _join(self, tokens):
        if self.mode == "char":
            return "".join(tokens)
        text = " ".join(tokens)
        text = re.sub(r"\s+([.,!?;:)\]%])", r"\1", text)    # no space before closing punctuation
        text = re.sub(r"([(\[])\s+", r"\1", text)            # no space after opening bracket
        return text

    def decode(self, ids, skip_padding=True):
        """list of ids -> readable text."""
        toks = [self.index_word.get(int(i), self.OOV) for i in ids]
        if skip_padding:
            toks = [t for t in toks if t != self.PAD]
        return self._join(toks)

    def decode_batch(self, array):
        """2-D array of ids -> list of texts (padding removed)."""
        return [self.decode(row) for row in array]

    def one_hot_decode(self, matrix):
        """one-hot matrix (or probabilities) -> text, using argmax on every row."""
        return self.decode(np.argmax(matrix, axis=1))

    # ---------- utilities ----------
    def to_json(self, path=None):
        d = {"mode": self.mode, "lower": self.lower, "word_index": self.word_index}
        s = json.dumps(d, indent=1)
        if path:
            open(path, "w").write(s)
        return s

    @classmethod
    def from_json(cls, s):
        d = json.loads(s)
        tp = cls(mode=d["mode"], lower=d["lower"])
        tp.word_index = d["word_index"]
        tp.index_word = {i: t for t, i in tp.word_index.items()}
        return tp

## 3. Create a corpus and build the vocabulary

In [3]:
corpus = [
    "The cat sat on the mat.",
    "The dog chased the cat!",
    "Dogs and cats don't always get along.",
    "Is the dog happy? Yes, the dog is very happy.",
    "Machine learning turns text into numbers; numbers feed neural networks.",
    "Data augmentation and transfer learning help when data is limited.",
]

tp = TextProcess(mode="word")
tp.fit(corpus)
print("Vocabulary size (incl. <PAD>, <OOV>):", tp.vocab_size)

vocab_df = pd.DataFrame({"token": list(tp.word_index.keys()), "id": list(tp.word_index.values())})
vocab_df["count"] = vocab_df["token"].map(tp.word_counts).fillna(0).astype(int)
vocab_df.head(15)

Vocabulary size (incl. <PAD>, <OOV>): 40


,token,id,count
0,<PAD>,0,0
1,<OOV>,1,0
2,the,2,6
3,.,3,5
4,dog,4,3
5,is,5,3
6,cat,6,2
7,and,7,2
8,happy,8,2
9,learning,9,2


## 4. Encoding – text → numbers

In [4]:
sentence = "The dog chased the cat!"
print("Original   :", sentence)
print("Tokens     :", tp.tokenize(sentence))
ids = tp.encode(sentence)
print("Encoded ids:", ids)
pd.DataFrame({"token": tp.tokenize(sentence), "id": ids}).T

Original   : The dog chased the cat!
Tokens     : ['the', 'dog', 'chased', 'the', 'cat', '!']
Encoded ids: [2, 4, 15, 2, 6, 16]


,0,1,2,3,4,5
token,the,dog,chased,the,cat,!
id,2,4,15,2,6,16


## 5. Decoding – numbers → readable text

In [5]:
decoded = tp.decode(ids)
print("Encoded ids:", ids)
print("Decoded    :", decoded)
print("Original (normalised):", tp.normalize(sentence))
print("Round trip identical? ", decoded == tp.normalize(sentence))

Encoded ids: [2, 4, 15, 2, 6, 16]
Decoded    : the dog chased the cat!
Original (normalised): the dog chased the cat!
Round trip identical?  True


In [6]:
# Round trip for EVERY sentence of the corpus
for s in corpus:
    enc = tp.encode(s); dec = tp.decode(enc)
    print(f"{s!r}\n   ids : {enc}\n   back: {dec!r}   -> {'OK' if dec == tp.normalize(s) else 'MISMATCH'}\n")

'The cat sat on the mat.'
   ids : [2, 6, 12, 13, 2, 14, 3]
   back: 'the cat sat on the mat.'   -> OK

'The dog chased the cat!'
   ids : [2, 4, 15, 2, 6, 16]
   back: 'the dog chased the cat!'   -> OK

"Dogs and cats don't always get along."
   ids : [17, 7, 18, 19, 20, 21, 22, 3]
   back: "dogs and cats don't always get along."   -> OK

'Is the dog happy? Yes, the dog is very happy.'
   ids : [5, 2, 4, 8, 23, 24, 25, 2, 4, 5, 26, 8, 3]
   back: 'is the dog happy? yes, the dog is very happy.'   -> OK

'Machine learning turns text into numbers; numbers feed neural networks.'
   ids : [27, 9, 28, 29, 30, 10, 31, 10, 32, 33, 34, 3]
   back: 'machine learning turns text into numbers; numbers feed neural networks.'   -> OK

'Data augmentation and transfer learning help when data is limited.'
   ids : [11, 35, 7, 36, 9, 37, 38, 11, 5, 39, 3]
   back: 'data augmentation and transfer learning help when data is limited.'   -> OK



## 6. More representations
### 6.1 Batches with padding
Neural networks need rectangular input, so shorter sequences are padded with `0` (`<PAD>`). Decoding skips the padding.

In [7]:
batch = ["The cat sat on the mat.", "Is the dog happy?", "Data is limited."]
X = tp.encode_batch(batch)
print("Padded integer matrix", X.shape)
print(X)
print("\nDecoded back:")
for t in tp.decode_batch(X): print("  ", t)

Padded integer matrix (3, 7)
[[ 2  6 12 13  2 14  3]
 [ 5  2  4  8 23  0  0]
 [11  5 39  3  0  0  0]]

Decoded back:
   the cat sat on the mat.
   is the dog happy?
   data is limited.


### 6.2 One-hot encoding
Each token id becomes a vector of length `vocab_size` with a single `1`. `one_hot_decode` finds the `1` with `argmax` and recovers the text.

In [8]:
oh = tp.one_hot_encode("the dog is happy")
print("One-hot matrix shape (tokens x vocab):", oh.shape)
print("Row sums (each token has exactly one 1):", oh.sum(axis=1))
print("Positions of the 1s (= token ids):    ", oh.argmax(axis=1).tolist())
print("Decoded from one-hot:", tp.one_hot_decode(oh))

One-hot matrix shape (tokens x vocab): (4, 40)
Row sums (each token has exactly one 1): [1. 1. 1. 1.]
Positions of the 1s (= token ids):     [2, 4, 5, 8]
Decoded from one-hot: the dog is happy


### 6.3 Unknown words (`<OOV>`)
Words that were not in the training corpus cannot be recovered – they are mapped to the reserved `<OOV>` id.

In [9]:
s = "The elephant sat on the mat."
e = tp.encode(s)
print("Encoded:", e)
print("Decoded:", tp.decode(e))

Encoded: [2, 1, 12, 13, 2, 14, 3]
Decoded: the <OOV> sat on the mat.


### 6.4 Character-level encoding
With `mode="char"` every character is a token, so the round trip is **exactly lossless** (even the capital letters if `lower=False`).

In [10]:
msg = "Hello, World! 123"
cp2 = TextProcess(mode="char", lower=False).fit(corpus + [msg])   # vocabulary containing every character
print("Char vocabulary size:", cp2.vocab_size)
print("Message :", msg)
print("Encoded :", cp2.encode(msg))
print("Decoded :", cp2.decode(cp2.encode(msg)))

Char vocabulary size: 42
Message : Hello, World! 123
Encoded : [37, 3, 12, 12, 9, 25, 2, 38, 9, 11, 12, 10, 23, 2, 39, 40, 41]
Decoded : Hello, World! 123


### 6.5 Saving and loading the vocabulary
A trained model must use the same vocabulary at inference time, so the mapping can be saved as JSON and restored.

In [11]:
saved = tp.to_json()
tp_loaded = TextProcess.from_json(saved)
print("Same vocabulary after reload:", tp_loaded.word_index == tp.word_index)
print("Encoding with reloaded object:", tp_loaded.encode("The dog chased the cat!"))

Same vocabulary after reload: True
Encoding with reloaded object: [2, 4, 15, 2, 6, 16]


### 6.6 Cross-check with Keras `TextVectorization`
Keras ships a built-in layer for the same job. Our own `TextProcess` produces the same kind of integer sequences (the exact ids can differ because Keras strips punctuation).

In [12]:
import tensorflow as tf
from tensorflow import keras

vec = keras.layers.TextVectorization(output_mode="int", standardize="lower_and_strip_punctuation")
vec.adapt(corpus)
vocab = vec.get_vocabulary()
out = vec(tf.constant(["the dog chased the cat"])).numpy()[0]
print("Keras ids  :", out.tolist())
print("Keras back :", " ".join(vocab[i] for i in out))

mine = TextProcess().fit(corpus)
print("TextProcess ids :", mine.encode("the dog chased the cat"))
print("TextProcess back:", mine.decode(mine.encode("the dog chased the cat")))

Keras ids  : [2, 4, 30, 2, 9]
Keras back : the dog chased the cat
TextProcess ids : [2, 4, 15, 2, 6]
TextProcess back: the dog chased the cat


## 7. Test suite – every required method is verified

In [13]:
results = []
def check(name, condition):
    results.append((name, bool(condition)))
    print(("PASS" if condition else "FAIL"), "-", name)
    assert condition, name

t = TextProcess(mode="word").fit(corpus)

# --- vocabulary ---
check("fit(): reserved ids <PAD>=0 and <OOV>=1", t.word_index["<PAD>"] == 0 and t.word_index["<OOV>"] == 1)
check("fit(): most frequent token ('the') gets the smallest real id (2)", t.word_index["the"] == 2)
check("fit(): index_word is the inverse of word_index", all(t.index_word[i] == w for w, i in t.word_index.items()))

# --- encoding ---
e = t.encode("The cat sat.")
check("encode(): returns a list of ints", isinstance(e, list) and all(isinstance(i, int) for i in e))
check("encode(): one id per token (the, cat, sat, .)", len(e) == 4)
check("encode(): case-insensitive", t.encode("THE CAT") == t.encode("the cat"))
check("encode(): unknown word -> OOV id", t.encode("zebra") == [1])
Xb = t.encode_batch(["the cat", "the dog chased the cat"])
check("encode_batch(): rectangular array padded with 0", Xb.shape == (2, 5) and Xb[0, 2:].tolist() == [0, 0, 0])
check("encode_batch(): maxlen truncates", t.encode_batch(["the dog chased the cat"], maxlen=3).shape == (1, 3))
oh = t.one_hot_encode("the cat")
check("one_hot_encode(): shape (tokens, vocab) and one 1 per row", oh.shape == (2, t.vocab_size) and (oh.sum(1) == 1).all())

# --- decoding ---
for s in corpus:
    check(f"decode(encode()) round trip: {s[:35]!r}", t.decode(t.encode(s)) == t.normalize(s))
check("decode(): padding is skipped", t.decode([2, 0, 0]) == "the")
check("decode_batch(): padded batch decodes to the original sentences",
      t.decode_batch(Xb) == ["the cat", "the dog chased the cat"])
check("one_hot_decode(): inverse of one_hot_encode", t.one_hot_decode(t.one_hot_encode("the dog is happy.")) == "the dog is happy.")
check("decode(): OOV id decodes to <OOV>", t.decode([1]) == "<OOV>")

# --- char mode + persistence ---
c = TextProcess(mode="char", lower=False).fit(["Hello, World! 123"])
check("char mode: exact lossless round trip", c.decode(c.encode("Hello, World! 123")) == "Hello, World! 123")
check("to_json()/from_json(): vocabulary restored", TextProcess.from_json(t.to_json()).encode("the dog") == t.encode("the dog"))

print(f"\n{sum(ok for _, ok in results)}/{len(results)} tests passed")

PASS - fit(): reserved ids <PAD>=0 and <OOV>=1
PASS - fit(): most frequent token ('the') gets the smallest real id (2)
PASS - fit(): index_word is the inverse of word_index
PASS - encode(): returns a list of ints
PASS - encode(): one id per token (the, cat, sat, .)
PASS - encode(): case-insensitive
PASS - encode(): unknown word -> OOV id
PASS - encode_batch(): rectangular array padded with 0
PASS - encode_batch(): maxlen truncates
PASS - one_hot_encode(): shape (tokens, vocab) and one 1 per row
PASS - decode(encode()) round trip: 'The cat sat on the mat.'
PASS - decode(encode()) round trip: 'The dog chased the cat!'
PASS - decode(encode()) round trip: "Dogs and cats don't always get alon"
PASS - decode(encode()) round trip: 'Is the dog happy? Yes, the dog is v'
PASS - decode(encode()) round trip: 'Machine learning turns text into nu'
PASS - decode(encode()) round trip: 'Data augmentation and transfer lear'
PASS - decode(): padding is skipped
PASS - decode_batch(): padded batch decodes 

## 8. Summary
* `TextProcess.encode / encode_batch / one_hot_encode` convert text into numerical representations (integer ids, padded matrices, one-hot vectors).
* `TextProcess.decode / decode_batch / one_hot_decode` convert the numbers back into readable text.
* Word-level encoding is lossless for words in the vocabulary (up to lower-casing); unseen words become `<OOV>`. Character-level encoding is fully lossless.
* All tests in Section 7 pass, so the class is complete and verified.